# Notebook 01 · Una DGAN que genera imágenes de dígitos

**Introducción al Deep Learning — Módulo 3, Unidad 3: GANs (Parte II)**

Damos el salto de los números a las **imágenes**. Construiremos una **DGAN** (GAN profunda) que aprende a
generar imágenes de dígitos (8×8) a partir de ruido:

- **Generador**: `Dense → Reshape → Conv2DTranspose → LeakyReLU → BatchNormalization`.
- **Discriminador**: una CNN de clasificación binaria (real / falso).

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.
> Entrenar una GAN es lento; con más épocas las imágenes mejoran (usa GPU en Colab).

## 1. Las imágenes reales

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
tf.random.set_seed(42); np.random.seed(42)

X = (load_digits().images.reshape(-1, 8, 8, 1).astype("float32") / 16.0)
print("Imágenes reales:", X.shape)

fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for ax, im in zip(axes, X[:8]):
    ax.imshow(im.reshape(8, 8), cmap="gray"); ax.axis("off")
fig.suptitle("Dígitos reales", y=1.1); plt.show()

## 2. El generador (de ruido a imagen)

`Dense` proyecta el ruido, `Reshape` lo convierte en una imagen pequeña y `Conv2DTranspose` la amplía hasta 8×8.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Reshape, Conv2DTranspose, Conv2D,
                                     LeakyReLU, BatchNormalization, Flatten, Input)

DIM_RUIDO = 16

generador = Sequential([
    Input(shape=(DIM_RUIDO,)),
    Dense(2 * 2 * 32), Reshape((2, 2, 32)),
    Conv2DTranspose(32, 3, strides=2, padding="same"), BatchNormalization(), LeakyReLU(0.2),  # 4x4
    Conv2DTranspose(16, 3, strides=2, padding="same"), BatchNormalization(), LeakyReLU(0.2),  # 8x8
    Conv2D(1, 3, padding="same", activation="sigmoid"),
], name="generador")
print("Salida del generador:", generador.output_shape)

## 3. El discriminador (real o falso)

In [ ]:
discriminador = Sequential([
    Input(shape=(8, 8, 1)),
    Conv2D(16, 3, strides=2, padding="same"), LeakyReLU(0.2),
    Conv2D(32, 3, strides=2, padding="same"), LeakyReLU(0.2),
    Flatten(),
    Dense(1, activation="sigmoid"),
], name="discriminador")

bce = tf.keras.losses.BinaryCrossentropy()
opt_g = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
opt_d = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
print("Discriminador listo.")

## 4. El bucle de entrenamiento adversario

In [ ]:
@tf.function
def paso(reales):
    n = tf.shape(reales)[0]
    z = tf.random.normal((n, DIM_RUIDO))
    with tf.GradientTape() as td:
        falsas = generador(z, training=True)
        loss_d = bce(tf.ones_like(discriminador(reales)), discriminador(reales, training=True)) \
               + bce(tf.zeros_like(discriminador(falsas)), discriminador(falsas, training=True))
    opt_d.apply_gradients(zip(td.gradient(loss_d, discriminador.trainable_variables),
                             discriminador.trainable_variables))
    z = tf.random.normal((n, DIM_RUIDO))
    with tf.GradientTape() as tg:
        falsas = generador(z, training=True)
        loss_g = bce(tf.ones_like(discriminador(falsas)), discriminador(falsas, training=True))
    opt_g.apply_gradients(zip(tg.gradient(loss_g, generador.trainable_variables),
                             generador.trainable_variables))
    return loss_d, loss_g

EPOCAS, BATCH = 60, 64
for e in range(EPOCAS):
    idx = np.random.permutation(len(X))
    for i in range(0, len(X), BATCH):
        ld, lg = paso(X[idx[i:i+BATCH]])
    if e % 15 == 0:
        print(f"época {e:2d} | loss_D {ld.numpy():.3f} | loss_G {lg.numpy():.3f}")
print("Entrenamiento terminado.")

## 5. Imágenes generadas por la red

In [ ]:
z = tf.random.normal((8, DIM_RUIDO))
generadas = generador(z, training=False).numpy()

fig, axes = plt.subplots(1, 8, figsize=(11, 1.6))
for ax, im in zip(axes, generadas):
    ax.imshow(im.reshape(8, 8), cmap="gray"); ax.axis("off")
fig.suptitle("Dígitos generados por la DGAN (a partir de ruido)", y=1.1); plt.show()

Con pocas épocas las imágenes son borrosas, pero ya muestran trazos parecidos a dígitos: la red
**genera imágenes nuevas desde ruido**. Con más épocas (y datos mayores como MNIST 28×28) la calidad
mejora mucho.

## Resumen

- El **generador** usa `Conv2DTranspose` para ampliar el ruido hasta una imagen.
- El **discriminador** es una CNN que separa reales de falsas.
- El **bucle adversario** alterna el entrenamiento de ambas redes.
- `LeakyReLU`, `BatchNormalization` y `Adam(beta_1=0.5)` estabilizan el entrenamiento.

➡️ En el **Notebook 02** haremos una **GAN condicional**: pedirle que genere un dígito concreto.